# 04 — Fine-tune + Phase 5.9 (RAG metrics + GitHub push)

Thin Kaggle driver around `scripts/run_phase59_pipeline.py`.

**Defaults are safe.** Flip flags only on a **Kaggle T4**.

| Flag | Default | Purpose |
|------|---------|---------|
| `RUN_TRAIN` | `False` | Unsloth QLoRA train |
| `MAX_STEPS` | `60` | Smoke steps |
| `RUN_RAG` | `False` | ST embeddings + generate eval |
| `PUBLISH_HF` | `False` | Needs `HF_TOKEN` secret |
| `PUSH_GITHUB` | `False` | Needs `GITHUB_TOKEN` secret (`repo` scope) |

Never invents metric percentages. Report fill only when `dry_run=false`.


## 0. Knobs


In [ ]:
from pathlib import Path
import os
import sys

RUN_TRAIN = False
MAX_STEPS = 60
RUN_RAG = False
PUBLISH_HF = False
PUSH_GITHUB = False

print("RUN_TRAIN", RUN_TRAIN, "MAX_STEPS", MAX_STEPS)
print("RUN_RAG", RUN_RAG, "PUBLISH_HF", PUBLISH_HF, "PUSH_GITHUB", PUSH_GITHUB)


## 1. Clone + path + installs


In [ ]:
IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)

if IN_KAGGLE:
    work = Path("/kaggle/working")
    repo = work / "earnings-call-research-assistant"
    if not (repo / "src" / "earnings_call_research_assistant" / "inference.py").exists():
        %cd /kaggle/working
        !git clone --depth 1 https://github.com/nuwanda94/earnings-call-research-assistant.git
        repo = work / "earnings-call-research-assistant"
    REPO = repo.resolve()
else:
    REPO = Path("..").resolve()
    if not (REPO / "src" / "earnings_call_research_assistant").is_dir():
        REPO = Path.cwd().resolve()

SRC = REPO / "src"
assert (SRC / "earnings_call_research_assistant" / "inference.py").exists(), SRC
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
print("REPO:", REPO)

if IN_KAGGLE:
    %pip install -q pyyaml rank_bm25
    if RUN_TRAIN or RUN_RAG:
        %pip install -q unsloth transformers accelerate bitsandbytes datasets trl peft sentence-transformers
    if PUBLISH_HF:
        %pip install -q huggingface_hub
print("ready")


## 2. Run Phase 5.9 pipeline


In [ ]:
cmd = [sys.executable, "scripts/run_phase59_pipeline.py", f"--max-steps={MAX_STEPS}"]
if RUN_TRAIN:
    cmd.append("--train")
if RUN_RAG:
    cmd.append("--rag")
if PUBLISH_HF:
    cmd.append("--publish-hf")
if PUSH_GITHUB:
    cmd.append("--push-github")
print("cmd:", cmd)
import subprocess
raise SystemExit(subprocess.call(cmd))


## Done

Artifacts (after a real `--rag` / `RUN_RAG=True` run):

- `evals/reports/rag_metrics.json`
- `evals/reports/rag_generation_metrics.json`
- `evals/reports/RAG_EVAL_REPORT.md`
- optional git push of those + README / PROGRESS

Set `RUN_TRAIN=True`, `RUN_RAG=True`, `PUSH_GITHUB=True` on T4 to close Phase 5.9.
